# Only 1 image with a mask, the others either didnt have masks or were corrupted links.

In [1]:
import sys, shutil
from pathlib import Path

PROJ = Path("/home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed")
sys.path.insert(0, str(PROJ))

from data_prep.prep_utils import DatasetConfig, run_prep

# Input directories
IMAGES_DIR = Path("/home/rbielski/Pre_Post_Stroke_Rehab/Aggregated_Images_and_masks/Images_with_masks")
MASKS_DIR  = Path("/home/rbielski/Pre_Post_Stroke_Rehab/Aggregated_Images_and_masks/Masks")

# Intermediate output (nested slug folders, xfm, etc.)
PREP_OUT = PROJ / "prep_intermediate"

# Final flat output directories
REG_IMAGES = PROJ / "Registered_Normalized_Images"
REG_MASKS  = PROJ / "Registered_Normalized_Masks"
REG_IMAGES.mkdir(parents=True, exist_ok=True)
REG_MASKS.mkdir(parents=True, exist_ok=True)

print("Images dir exists:", IMAGES_DIR.exists(), f"({len(list(IMAGES_DIR.glob('*.nii.gz')))} files)")
print("Masks dir exists: ", MASKS_DIR.exists(),  f"({len(list(MASKS_DIR.glob('*.nii.gz')))} files)")


Images dir exists: True (1 files)
Masks dir exists:  True (1 files)


In [2]:
ds = DatasetConfig(
    name="PrePostStrokeRehab",
    images_dir=IMAGES_DIR,
    masks_dir=MASKS_DIR,
    t1_glob="*.nii.gz",
    mask_glob="*.nii.gz",
    skull_strip=True,
)

outputs = run_prep([ds], out_root=PREP_OUT)
print("\nPrep complete. Dataset roots:", outputs)


[PrePostStrokeRehab] image root: /home/rbielski/Pre_Post_Stroke_Rehab/Aggregated_Images_and_masks/Images_with_masks | mask root: /home/rbielski/Pre_Post_Stroke_Rehab/Aggregated_Images_and_masks/Masks
[PrePostStrokeRehab] globs: t1=*.nii.gz masks=*.nii.gz
[PrePostStrokeRehab] images: 1 masks: 1 pairs found: 1


2026-02-20 16:47:40.820993: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-20 16:47:42.110753: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE: forward compatibility was attempted on non supported HW
2026-02-20 16:47:42.110778: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-02-20 16:47:42.110782: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: jmgpu1
2026-02-20 16:47:42.1

  [skull_strip] sub-01_ses-pre_T1w.nii.gz → sub-01_ses-pre_T1w_brain.nii.gz
>> /home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed/tools/ants/bin/antsRegistration -d 3 -r [/home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed/prep_intermediate/PrePostStrokeRehab-Images-with-masks-bf596377/skull_stripped/sub-01_ses-pre_T1w_brain.nii.gz,1] -m Mattes[/home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed/prep_intermediate/PrePostStrokeRehab-Images-with-masks-bf596377/skull_stripped/sub-01_ses-pre_T1w_brain.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed/dat

In [3]:
# Collect normalized T1s and cleaned masks into flat final output directories
for ds_root in outputs:
    t1_norm_dir = ds_root / "mni_1mm_ants_fixed" / "t1_norm"
    masks_clean_dir = ds_root / "mni_1mm_ants_fixed" / "masks_clean"

    t1s = sorted(t1_norm_dir.glob("*.nii.gz"))
    masks = sorted(masks_clean_dir.glob("*.nii.gz"))

    for f in t1s:
        shutil.copy2(f, REG_IMAGES / f.name)
    for f in masks:
        shutil.copy2(f, REG_MASKS / f.name)

    print(f"Copied {len(t1s)} normalized T1s  → {REG_IMAGES}")
    print(f"Copied {len(masks)} cleaned masks  → {REG_MASKS}")

print("\nDone!")


Copied 1 normalized T1s  → /home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed/Registered_Normalized_Images
Copied 1 cleaned masks  → /home/rbielski/stroke_cleaned/Pre_Post_Stroke_Rehab_Processed/Registered_Normalized_Masks

Done!


In [ ]:
from data_prep.viewer import show_viewer_dirs

show_viewer_dirs(
    images_dir=REG_IMAGES,
    masks_dir=REG_MASKS,
)


Output()